## 8.2 GPT for code dictation

## Import Required Libraries

In this section we import the main libraries required to train and analyze a GPT-2 model.

Libraries used:

- **torch** – The deep learning framework used to run the model.
- **pandas** – Used to manipulate and analyze tabular data.
- **transformers** – Hugging Face library that provides pretrained models such as GPT-2.
- **bertviz** – A visualization tool used to inspect the attention mechanism inside transformer models.

These libraries allow us to load the model, prepare the data, train the model, and inspect its behavior.

In [13]:
from transformers import GPT2Tokenizer, DataCollatorForLanguageModeling, TrainingArguments, Trainer, \
                         GPT2LMHeadModel, pipeline
from datasets import Dataset
import pandas as pd

## Load the GPT-2 Tokenizer

Here we load the tokenizer associated with the GPT-2 model.

A **tokenizer** converts raw text into numerical tokens that the model can understand.

Example:

Text:
"Hello world"

Token IDs:
[15496, 995]

The tokenizer ensures that the text input is transformed into the same token representation used during the original GPT-2 training.

In [14]:
data = pd.read_csv('../data/english_to_latex.csv')

print(data.shape)

data.head(2)

(50, 2)


,English,LaTeX
0,integral from a to b of x squared,"\int_{a}^{b} x^2 \,dx"
1,integral from negative 1 to 1 of x squared,"\int_{-1}^{1} x^2 \,dx"


## Define the Task Prompt

We define a prompt that informs the model about the task we want it to learn.

CONVERSION_PROMPT = "LCT\n"

LCT stands for **LaTeX Conversion Task**.

This prompt acts as a signal that the model should perform a specific task: converting English descriptions into LaTeX mathematical expressions.

In [15]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token

# Add our singular prompt
CONVERSION_PROMPT = 'LCT\n'  # LaTeX conversion task

CONVERSION_TOKEN = 'LaTeX:'


loading file https://huggingface.co/gpt2/resolve/main/vocab.json from cache at /Users/sinanozdemir/.cache/huggingface/transformers/684fe667923972fb57f6b4dcb61a3c92763ad89882f3da5da9866baf14f2d60f.c7ed1f96aac49e745788faa77ba0a26a392643a50bb388b9c04ff469e555241f
loading file https://huggingface.co/gpt2/resolve/main/merges.txt from cache at /Users/sinanozdemir/.cache/huggingface/transformers/c0c761a63004025aeadd530c4c27b860ec4ecbe8a00531233de21d865a402598.5d12962c5ee615a4c803841266e9c3be9a691a924f72d395d3a6c6c81157788b
loading file https://huggingface.co/gpt2/resolve/main/added_tokens.json from cache at None
loading file https://huggingface.co/gpt2/resolve/main/special_tokens_map.json from cache at None
loading file https://huggingface.co/gpt2/resolve/main/tokenizer_config.json from cache at None
loading file https://huggingface.co/gpt2/resolve/main/tokenizer.json from cache at /Users/sinanozdemir/.cache/huggingface/transformers/16a2f78023c8dc511294f0c97b5e10fde3ef9889ad6d11ffaa2a00714e73

## Construct Training Examples

Here we construct training examples by combining the prompt, the English description, and the target LaTeX output.

Each training sample follows this structure:

LCT  
English: <English sentence>  
LaTeX: <LaTeX expression>

Example:

LCT  
English: integral from a to b of x squared  
LaTeX: \int_{a}^{b} x^2 dx

This format teaches the model the relationship between the English description and its corresponding LaTeX representation.

In [122]:
# This is our "training prompt" that we want GPT2 to recognize and learn
training_examples = f'{CONVERSION_PROMPT}English: ' + data['English'] + '\n' + CONVERSION_TOKEN + ' ' + data['LaTeX'].astype(str)

print(training_examples[0])


LCT
English: integral from a to b of x squared
LaTeX: \int_{a}^{b} x^2 \,dx


## Convert Training Examples into a DataFrame

We store the constructed training examples inside a Pandas DataFrame.

The DataFrame contains a single column called "text", where each row is a formatted training example.

This structure allows us to easily convert the data into a dataset for model training.

In [123]:
task_df = pd.DataFrame({'text': training_examples})

task_df.head(2)

,text
0,LCT\nEnglish: integral from a to b of x square...
1,LCT\nEnglish: integral from negative 1 to 1 of...


## Tokenize the Dataset

Before training, the text must be converted into token IDs.

The tokenizer splits the text into tokens and maps each token to its corresponding numerical ID in the GPT-2 vocabulary.

This step prepares the dataset for training by converting raw text into numerical representations the model can process.

In [126]:
latex_data = Dataset.from_pandas(task_df)  # turn a pandas DataFrame into a Dataset

def preprocess(examples):  # tokenize our text but don't pad because our collator will pad for us dynamically
    return tokenizer(examples['text'], truncation=True)

latex_data = latex_data.map(preprocess, batched=True)

latex_data = latex_data.train_test_split(train_size=.8)

  0%|          | 0/1 [00:00<?, ?ba/s]

In [127]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [128]:
latex_gpt2 = GPT2LMHeadModel.from_pretrained('gpt2')

loading configuration file https://huggingface.co/gpt2/resolve/main/config.json from cache at /Users/sinanozdemir/.cache/huggingface/transformers/fc674cd6907b4c9e933cb42d67662436b89fa9540a1f40d7c919d0109289ad01.7d2e0efa5ca20cef4fb199382111e9d3ad96fd77b849e1d4bed13a66e1336f51
Model config GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-gen

## Data Collator for Language Modeling

The DataCollator is responsible for preparing batches during training.

Its responsibilities include:

- Padding sequences to equal lengths
- Creating the input tensors
- Preparing labels for next-token prediction

Since GPT-2 is a **causal language model**, it predicts the next token in a sequence.

## Training Configuration

TrainingArguments define how the model should be trained.

Important parameters include:

- **output_dir** – directory where model checkpoints are saved
- **num_train_epochs** – number of times the dataset is processed during training
- **batch size** – number of samples processed at once
- **logging_steps** – frequency of training logs
- **evaluation_strategy** – when evaluation occurs
- **save_strategy** – when model checkpoints are saved

In [65]:
training_args = TrainingArguments(
    output_dir="./english_to_latex",
    overwrite_output_dir=True, #overwrite the content of the output directory
    num_train_epochs=10, # number of training epochs
    per_device_train_batch_size=2, # batch size for training
    per_device_eval_batch_size=20,  # batch size for evaluation
    load_best_model_at_end=True,
    logging_steps=5,
    log_level='info',
    evaluation_strategy='epoch',
    save_strategy='epoch'
)

trainer = Trainer(
    model=latex_gpt2,
    args=training_args,
    train_dataset=latex_data["train"],
    eval_dataset=latex_data["test"],
    data_collator=data_collator,
)

trainer.evaluate()

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
The following columns in the evaluation set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text.
***** Running Evaluation *****
  Num examples = 10
  Batch size = 20


{'eval_loss': 4.891345977783203,
 'eval_runtime': 0.6281,
 'eval_samples_per_second': 15.921,
 'eval_steps_per_second': 1.592}

## Train the Model

The training process begins with:

trainer.train()

During training:

1. The model receives an input sequence.
2. It predicts the next token.
3. The prediction is compared with the true token.
4. The loss is calculated.
5. Model weights are updated using backpropagation.

This process repeats for multiple epochs until the model learns the mapping between English descriptions and LaTeX expressions.

In [66]:
trainer.train()

The following columns in the training set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text.
***** Running training *****
  Num examples = 40
  Num Epochs = 10
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 2
  Gradient Accumulation steps = 1
  Total optimization steps = 200


Epoch,Training Loss,Validation Loss
1,1.457600,1.135592
2,0.773500,0.939705
3,0.765400,0.835251
4,0.618300,0.797922
5,0.548200,0.902167
6,0.572200,0.815490
7,0.381500,0.843624
8,0.361600,0.889109
9,0.399900,0.878657
10,0.302300,0.889017


The following columns in the evaluation set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text.
***** Running Evaluation *****
  Num examples = 10
  Batch size = 20
Saving model checkpoint to ./english_to_latex/checkpoint-20
Configuration saved in ./english_to_latex/checkpoint-20/config.json
Model weights saved in ./english_to_latex/checkpoint-20/pytorch_model.bin
The following columns in the evaluation set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text.
***** Running Evaluation *****
  Num examples = 10
  Batch size = 20
Saving model checkpoint to ./english_to_latex/checkpoint-40
Configuration saved in ./english_to_latex/checkpoint-40/config.json
Model weights saved in ./english_to_latex/checkpoint-40/pytorch_model.bin
The following columns in the evaluation set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text.
***** Running Evaluation *****
  Num exam

TrainOutput(global_step=200, training_loss=0.7566462355852127, metrics={'train_runtime': 218.1877, 'train_samples_per_second': 1.833, 'train_steps_per_second': 0.917, 'total_flos': 6087287808000.0, 'train_loss': 0.7566462355852127, 'epoch': 10.0})

## Evaluate Model Performance

After training, the model is evaluated on a validation dataset.

Evaluation measures how well the model generalizes to unseen examples.

Common metrics include loss and perplexity.

In [67]:
trainer.evaluate()

The following columns in the evaluation set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text.
***** Running Evaluation *****
  Num examples = 10
  Batch size = 20


{'eval_loss': 0.7979221343994141,
 'eval_runtime': 1.0997,
 'eval_samples_per_second': 9.093,
 'eval_steps_per_second': 0.909,
 'epoch': 10.0}

In [42]:
# Calculus Made Easy by Silvanus P. Thompson - https://gutenberg.org/ebooks/33283

calculus_data = TextDataset(
    tokenizer=tokenizer,
    file_path='../data/calculus made easy.txt',  # Principles of Data Science - Sinan Ozdemir
    block_size=32
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False,  # MLM is Masked Language Modelling
)

latex_gpt2 = GPT2LMHeadModel.from_pretrained('gpt2')

training_args = TrainingArguments(
    output_dir="./calculus",
    overwrite_output_dir=True, #overwrite the content of the output directory
    num_train_epochs=1, # number of training epochs
    per_device_train_batch_size=32, # batch size for training
    per_device_eval_batch_size=32,  # batch size for evaluation
    load_best_model_at_end=True,
    logging_steps=50,
    eval_steps=50,
    evaluation_strategy='steps',
    save_strategy='steps'
)

trainer = Trainer(
    model=latex_gpt2,
    args=training_args,
    data_collator=data_collator,
    train_dataset=calculus_data.examples[:int(len(calculus_data.examples)*.8)],
    eval_dataset=calculus_data.examples[int(len(calculus_data.examples)*.8):]
)

Loading features from cached file ../data/cached_lm_GPT2Tokenizer_32_calculus made easy.txt [took 0.016 s]
loading configuration file https://huggingface.co/gpt2/resolve/main/config.json from cache at /Users/sinanozdemir/.cache/huggingface/transformers/fc674cd6907b4c9e933cb42d67662436b89fa9540a1f40d7c919d0109289ad01.7d2e0efa5ca20cef4fb199382111e9d3ad96fd77b849e1d4bed13a66e1336f51
Model config GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels":

In [43]:
trainer.evaluate()  # initial loss for the calculus book

***** Running Evaluation *****
  Num examples = 1624
  Batch size = 32


{'eval_loss': 2.5129024982452393,
 'eval_runtime': 72.3714,
 'eval_samples_per_second': 22.44,
 'eval_steps_per_second': 0.705}

In [44]:
trainer.train()

***** Running training *****
  Num examples = 6494
  Num Epochs = 1
  Instantaneous batch size per device = 32
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 1
  Total optimization steps = 203


Step,Training Loss,Validation Loss
50,1.796800,1.640845
100,1.581200,1.589910
150,1.567200,1.567876
200,1.466200,1.558849


***** Running Evaluation *****
  Num examples = 1624
  Batch size = 32
***** Running Evaluation *****
  Num examples = 1624
  Batch size = 32
***** Running Evaluation *****
  Num examples = 1624
  Batch size = 32
***** Running Evaluation *****
  Num examples = 1624
  Batch size = 32


Training completed. Do not forget to share your model on huggingface.co/models =)




TrainOutput(global_step=203, training_loss=1.6039342269521628, metrics={'train_runtime': 1292.6277, 'train_samples_per_second': 5.024, 'train_steps_per_second': 0.157, 'total_flos': 106051903488000.0, 'train_loss': 1.6039342269521628, 'epoch': 1.0})

In [45]:
trainer.save_model()

Saving model checkpoint to ./calculus
Configuration saved in ./calculus/config.json
Model weights saved in ./calculus/pytorch_model.bin


In [73]:
calculus_latex_gpt2 = GPT2LMHeadModel.from_pretrained('./calculus')  # load up our gpt pre-trained on calculus

training_args = TrainingArguments(
    output_dir="./calculus_english_to_latex",
    overwrite_output_dir=True, #overwrite the content of the output directory
    num_train_epochs=10, # number of training epochs
    per_device_train_batch_size=2, # batch size for training
    per_device_eval_batch_size=20,  # batch size for evaluation
    load_best_model_at_end=True,
    logging_steps=5,
    log_level='info',
    evaluation_strategy='epoch',
    save_strategy='epoch'
)

trainer = Trainer(
    model=calculus_latex_gpt2,
    args=training_args,
    train_dataset=latex_data["train"],
    eval_dataset=latex_data["test"],
    data_collator=data_collator,
)

trainer.evaluate()  # loss is starting slightly lower than before

loading configuration file ./calculus/config.json
Model config GPT2Config {
  "_name_or_path": "gpt2",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "torch_dtype": "float32",
  "transformers_version": "4.15.0",
  "use_cache": true,
  "vocab_size": 50257
}

loading weights fi

{'eval_loss': 4.565759181976318,
 'eval_runtime': 0.6014,
 'eval_samples_per_second': 16.627,
 'eval_steps_per_second': 1.663}

In [74]:
trainer.train()

The following columns in the training set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text.
***** Running training *****
  Num examples = 40
  Num Epochs = 10
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 2
  Gradient Accumulation steps = 1
  Total optimization steps = 200


Epoch,Training Loss,Validation Loss
1,1.357300,1.128648
2,0.694700,0.932949
3,0.742900,0.899620
4,0.602400,0.789026
5,0.513100,0.943910
6,0.566900,0.944910
7,0.406300,0.889982
8,0.363100,0.927460
9,0.406200,0.949464
10,0.299400,0.962857


The following columns in the evaluation set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text.
***** Running Evaluation *****
  Num examples = 10
  Batch size = 20
Saving model checkpoint to ./calculus_english_to_latex/checkpoint-20
Configuration saved in ./calculus_english_to_latex/checkpoint-20/config.json
Model weights saved in ./calculus_english_to_latex/checkpoint-20/pytorch_model.bin
The following columns in the evaluation set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text.
***** Running Evaluation *****
  Num examples = 10
  Batch size = 20
Saving model checkpoint to ./calculus_english_to_latex/checkpoint-40
Configuration saved in ./calculus_english_to_latex/checkpoint-40/config.json
Model weights saved in ./calculus_english_to_latex/checkpoint-40/pytorch_model.bin
The following columns in the evaluation set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ig

TrainOutput(global_step=200, training_loss=0.7141165947914123, metrics={'train_runtime': 230.0817, 'train_samples_per_second': 1.739, 'train_steps_per_second': 0.869, 'total_flos': 6087287808000.0, 'train_loss': 0.7141165947914123, 'epoch': 10.0})

In [75]:
trainer.evaluate()  # pre-training on the calculus book for one epoch led to a minor drop in loss

The following columns in the evaluation set  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text.
***** Running Evaluation *****
  Num examples = 10
  Batch size = 20


{'eval_loss': 0.7890258431434631,
 'eval_runtime': 0.7324,
 'eval_samples_per_second': 13.655,
 'eval_steps_per_second': 1.365,
 'epoch': 10.0}

## Save the Trained Model

After the training process is complete, we save the trained model using:

`trainer.save_model()`

This function stores the model weights and configuration files in the directory specified in `TrainingArguments` (for example: `./gpt2_pds`).

Saving the model is important because it allows us to:

- Reuse the trained model later without retraining
- Load the model for inference or testing
- Share the trained model with others
- Deploy the model in an application

The saved model can later be loaded using:

```python
from transformers import GPT2LMHeadModel
model = GPT2LMHeadModel.from_pretrained("./gpt2_pds")

In [76]:
trainer.save_model()  # save this model

Saving model checkpoint to ./calculus_english_to_latex
Configuration saved in ./calculus_english_to_latex/config.json
Model weights saved in ./calculus_english_to_latex/pytorch_model.bin


## Load the Trained Model and Create a Text Generation Pipeline

In this step, we load the previously trained GPT-2 model from disk and create a text generation pipeline.

First, we load the saved model using:

`GPT2LMHeadModel.from_pretrained('./calculus_english_to_latex')`

This command restores the trained model weights and configuration from the directory where the model was saved after training.

Next, we create a **text generation pipeline** using Hugging Face's `pipeline` utility:

`pipeline('text-generation', model=loaded_model, tokenizer=tokenizer)`

The pipeline simplifies the process of generating text from the model. It automatically handles:

- tokenizing the input text
- passing the tokens to the model
- generating predictions
- decoding the output tokens back into readable text

This allows us to easily generate LaTeX expressions from English descriptions using the trained GPT-2 model.

In [144]:
loaded_model = GPT2LMHeadModel.from_pretrained('./calculus_english_to_latex')
latex_generator = pipeline('text-generation', model=loaded_model, tokenizer=tokenizer)

loading configuration file ./calculus_english_to_latex/config.json
Model config GPT2Config {
  "_name_or_path": "./calculus",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "torch_dtype": "float32",
  "transformers_version": "4.15.0",
  "use_cache": true,
  "vocab_size": 5025

In [149]:
text_sample = 'f of x equals integral from 0 to pi of x to the fourth power'
conversion_text_sample = f'{CONVERSION_PROMPT}English: {text_sample}\n{CONVERSION_TOKEN}'

print(conversion_text_sample)

LCT
English: f of x equals integral from 0 to pi of x to the fourth power
LaTeX:
